In [ ]:
import torch, sys, os
print('Torch', torch.__version__, 'CUDA available:', torch.cuda.is_available())
PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

In [ ]:
# Copy attached dataset if present
import shutil
KAGGLE_DATASET_PATH = '/kaggle/input/yoruba-parallel'
if os.path.exists(KAGGLE_DATASET_PATH):
    for fname in os.listdir(KAGGLE_DATASET_PATH):
        src = os.path.join(KAGGLE_DATASET_PATH, fname)
        dst = os.path.join(DATA_DIR, fname)
        if os.path.isfile(src):
            shutil.copy(src, dst)
else:
    print('No attached Kaggle dataset found at', KAGGLE_DATASET_PATH)

In [ ]:
# Sanity check
try:
    from scripts.sanity_check import main as sanity_main
    sanity_main()
except Exception as e:
    print('sanity_check failed:', e)

## Train Transformer (short run)
cells
metadata
source

Self-contained: installs requirements if present, sets seeds and thread limits, copies the attached Kaggle dataset into `./data/`, runs a short smoke-training job, and evaluates + saves predictions.

In [ ]:
import os, sys, torch, random, numpy as np
print('Torch', torch.__version__, 'CUDA available:', torch.cuda.is_available())
PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'
torch.set_num_threads(4)
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
print('Setup complete')

In [ ]:
# Install requirements if provided
import subprocess
req = os.path.join(PROJECT_ROOT, 'requirements.txt')
if os.path.exists(req):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', req], check=False)
else:
    print('No requirements.txt; assuming Kaggle environment')

In [ ]:
# Copy attached dataset into ./data/ if available
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
import shutil
KAGGLE_DATASET_PATH = '/kaggle/input/yoruba-parallel'
if os.path.exists(KAGGLE_DATASET_PATH):
    os.makedirs(DATA_DIR, exist_ok=True)
    for fname in os.listdir(KAGGLE_DATASET_PATH):
        src = os.path.join(KAGGLE_DATASET_PATH, fname)
        dst = os.path.join(DATA_DIR, fname)
        if os.path.isfile(src):
            shutil.copy(src, dst)
else:
    print('No attached Kaggle dataset found at', KAGGLE_DATASET_PATH)

In [ ]:
# Sanity check
try:
    from scripts.sanity_check import main as sanity_main
    sanity_main()
except Exception as e:
    print('sanity_check failed:', e)

## Train Transformer (short run)
Run this short training job for a smoke test; increase epochs for serious experiments.

In [ ]:
from types import SimpleNamespace
from src.train_transformer import train
device = 'cuda' if torch.cuda.is_available() else 'cpu'
args = SimpleNamespace(
    data_dir=DATA_DIR,
    epochs=2,
    batch_size=32,
    output_dir=MODELS_DIR,
    lr=5e-4,
    num_workers=2,
    accum_steps=1,
    num_threads=4,
    no_eval=True,
    eval_subset=0,
    optimizer='adamw',
    weight_decay=0.01,
    clip_norm=1.0,
    ss_start=1.0,
    ss_end=1.0,
    ss_anneal_epochs=0
)
print('Training Transformer on device', device)
train(args)
print('Done')

In [ ]:
# Evaluate and save predictions
ckpt = os.path.join(MODELS_DIR, 'transformer_last.pt')
if os.path.exists(ckpt):
    out_csv = os.path.join(MODELS_DIR, 'transformer_preds.csv')
    print('Evaluating', ckpt, 'and saving preds to', out_csv)
    os.system(f"python .\scripts\eval_dev.py --data_dir {DATA_DIR} --model_path {ckpt} --model_type transformer --eval_subset 0 --batch_size 8 --num_examples 20 --save_preds --preds_path {out_csv} --device {device}")
else:
    print('No checkpoint found at', ckpt)